In [1]:
from pathlib import Path
import json

# Our directories
sections_dir = Path("temporary_sections")
responses_dir = Path("../../data/test/raw_api_responses")

# Get all temporary audio sections
audio_files = sorted(sections_dir.glob("*.wav"))

# Get names of sections already transcribed
completed = {
    file.stem
    for file in responses_dir.glob("section_*.response")
}

# Find sections that still need transcription
pending = [
    file for file in audio_files
    if file.stem not in completed
]

print("Total audio sections:", len(audio_files))
print("Already transcribed:", len(audio_files) - len(pending))
print("Still need transcription:", len(pending))

print("\nFirst 5 pending sections:")
for file in pending[:5]:
    print(file.name)

Total audio sections: 10
Already transcribed: 10
Still need transcription: 0

First 5 pending sections:


In [2]:
import subprocess

original_audio = Path("../../data/test/test_audio.mp3")

def get_duration(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path)
        ],
        capture_output=True,
        text=True,
        check=True
    )

    return float(result.stdout.strip())

if original_audio.exists():
    duration = get_duration(original_audio)

    print("Original duration:", round(duration, 2), "seconds")
    print("Original duration:", round(duration / 60, 2), "minutes")
else:
    print("Original audio not found:", original_audio)

Original duration: 591.73 seconds
Original duration: 9.86 minutes


In [3]:
metadata_path = Path("temporary_sections.json")

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, list):
    print("Number of entries:", len(metadata))
    print(json.dumps(metadata[:2], indent=2, ensure_ascii=False))

elif isinstance(metadata, dict):
    print("Top-level keys:", list(metadata.keys()))
    print(
        json.dumps(metadata, indent=2, ensure_ascii=False)[:2000]
    )

Metadata type: list
Number of entries: 10
[
  {
    "filename": "section_000.wav",
    "start": 0,
    "end": 59.9
  },
  {
    "filename": "section_001.wav",
    "start": 58.9,
    "end": 122.55
  }
]


In [4]:
import re
from rapidfuzz import fuzz

# Read the entire book
book = Path("../../data/test/book.txt").read_text(
    encoding="utf-8"
)

# Normalize text for matching
def normalize(text):
    text = text.lower()

    for mark in ["‘", "’", "ʻ", "ʼ", "`"]:
        text = text.replace(mark, "'")

    text = re.sub(r"[^\w\s']", " ", text)
    return " ".join(text.split())


book_words = normalize(book).split()

# Read all existing transcription responses
response_files = sorted(
    responses_dir.glob("section_*.response")
)

matches = []
previous_start = 0
previous_end = 0

for index, response_path in enumerate(response_files):

    data = json.loads(
        response_path.read_text(encoding="utf-8")
    )

    transcription = " ".join(
        segment["text"].strip()
        for segment in data["segments"]
    )

    spoken_text = normalize(transcription)
    word_count = len(spoken_text.split())

    if word_count == 0:
        print("Empty transcription:", response_path.name)
        continue

    # Allow different book passage lengths
    window_sizes = [
        max(1, int(word_count * factor))
        for factor in [0.8, 1.0, 1.2]
    ]

    # First section: search the entire book
    if index == 0:
        search_start = 0
        search_end = len(book_words) - 1

    # Other sections: search near previous match
    else:
        search_start = max(0, previous_start - 30)
        search_end = min(
            len(book_words) - 1,
            previous_end + word_count * 3
        )

    best_score = -1
    best_start = 0
    best_end = 0

    # Search in small steps
    step = max(1, word_count // 10)

    for start in range(search_start, search_end + 1, step):

        for size in window_sizes:

            end = min(start + size, len(book_words))

            candidate = " ".join(book_words[start:end])

            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    # Refine the best position word by word
    for start in range(
        max(search_start, best_start - step),
        min(search_end, best_start + step) + 1
    ):
        for size in window_sizes:

            end = min(start + size, len(book_words))
            candidate = " ".join(book_words[start:end])
            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    previous_start = best_start
    previous_end = best_end

    matched_text = " ".join(
        book_words[best_start:best_end]
    )

    matches.append({
        "section": response_path.stem,
        "start_word": best_start,
        "end_word": best_end,
        "similarity": round(best_score, 2),
        "transcription": transcription,
        "book_text": matched_text,
        "needs_review": best_score < 85
    })

    print(
        response_path.stem,
        "| Score:", round(best_score, 2),
        "| Words:", best_start, "-", best_end
    )

# Save matching results
output_path = Path("../../data/test/book_matches_draft.json")

output_path.write_text(
    json.dumps(matches, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nSaved:", output_path)

section_000 | Score: 82.35 | Words: 0 - 51
section_001 | Score: 98.65 | Words: 51 - 151
section_002 | Score: 95.03 | Words: 150 - 237
section_003 | Score: 98.26 | Words: 240 - 330
section_004 | Score: 96.54 | Words: 331 - 452
section_005 | Score: 98.93 | Words: 453 - 550
section_006 | Score: 97.39 | Words: 550 - 642
section_007 | Score: 98.91 | Words: 640 - 738
section_008 | Score: 93.84 | Words: 738 - 823
section_009 | Score: 94.55 | Words: 828 - 890

Saved: ../../data/test/book_matches_draft.json


In [5]:
# Load the matching results
matches_path = Path(
    "../../data/test/book_matches_draft.json"
)

matches = json.loads(
    matches_path.read_text(encoding="utf-8")
)

print("Total matched sections:", len(matches))

previous = None
review_indices = []

for i, match in enumerate(matches):

    start = match["start_word"]
    end = match["end_word"]
    score = match["similarity"]

    issues = []

    # Check similarity
    if score < 85:
        issues.append("LOW SIMILARITY")

    # Check valid word positions
    if end <= start:
        issues.append("INVALID WORD RANGE")

    # Compare with previous section
    if previous is not None:

        previous_start = previous["start_word"]
        previous_end = previous["end_word"]

        gap = start - previous_end

        if start < previous_start:
            issues.append("BACKWARD MATCH")

        if gap > 40:
            issues.append("POSSIBLE MISSING TEXT")

    else:
        gap = None

    if issues:
        review_indices.append(i)

    print(f"\nSection: {match['section']}")
    print(f"Similarity: {score}%")
    print(f"Book words: {start} - {end}")
    print(f"Gap from previous: {gap}")
    print(f"Issues: {issues if issues else 'None detected'}")

    previous = match

print("\nSections requiring review:", len(review_indices))

Total matched sections: 10

Section: section_000
Similarity: 82.35%
Book words: 0 - 51
Gap from previous: None
Issues: ['LOW SIMILARITY']

Section: section_001
Similarity: 98.65%
Book words: 51 - 151
Gap from previous: 0
Issues: None detected

Section: section_002
Similarity: 95.03%
Book words: 150 - 237
Gap from previous: -1
Issues: None detected

Section: section_003
Similarity: 98.26%
Book words: 240 - 330
Gap from previous: 3
Issues: None detected

Section: section_004
Similarity: 96.54%
Book words: 331 - 452
Gap from previous: 1
Issues: None detected

Section: section_005
Similarity: 98.93%
Book words: 453 - 550
Gap from previous: 1
Issues: None detected

Section: section_006
Similarity: 97.39%
Book words: 550 - 642
Gap from previous: 0
Issues: None detected

Section: section_007
Similarity: 98.91%
Book words: 640 - 738
Gap from previous: -2
Issues: None detected

Section: section_008
Similarity: 93.84%
Book words: 738 - 823
Gap from previous: 0
Issues: None detected

Section: sec

In [6]:
for i in review_indices[:5]:

    match = matches[i]

    print("\n" + "=" * 60)
    print("SECTION:", match["section"])

    print("\nTRANSCRIPTION:")
    print(match["transcription"][:500])

    print("\nMATCHED BOOK:")
    print(match["book_text"][:500])


SECTION: section_000

TRANSCRIPTION:
Qadrli bolajonlar, o'zbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali. Qadim zamonda bir podsho o'tgan ekan, U taxtda doim g'amgin o'tirar ekan. Kunlarning birida o'ng qo'li vaziri undan: "Podshohim, nimagadir mudom g'amginsiz, boisi nima?" deb so'rabdi. "Mol-u davlatim yetarli, ammo o'lgandan keyin bel bog'lab bo'zlaydigan o'g'lim yo'q, o'g'il ko'rsam, boshqa armonim qolmaydi", debdi podsho. "Olampanoh, sizga maslahatim bor.

MATCHED BOOK:
ain ochilar y adim zamonda bir podsho o'tgan ekan u taxtda doim g'amgin o'tirarkan kunlarning birida o'ng 401 vaziri undan podshohim nimagadir mudom фатат 512 boisi nima deb so'rabdi mol u davlatim yetarli ammo o'lgandan keyin bel bog'lab bo'zlaydigan o'g'lim yo'q o'g'il ko'rsam boshqa armonim qolmaydi debdi podsho olampanoh sizga maslahatim


In [7]:
# Prepare all matched sections
alignment_plan = []

previous_match = None

for match in matches:

    section = match["section"]
    score = match["similarity"]

    audio_path = sections_dir / f"{section}.wav"

    issues = []

    # Check similarity
    if score < 85:
        issues.append("Low similarity")

    # Check audio file
    if not audio_path.exists():
        issues.append("Audio file missing")

    # Check matching positions
    if match["end_word"] <= match["start_word"]:
        issues.append("Invalid word range")

    # Check book progression
    if previous_match is not None:
        if match["start_word"] < previous_match["start_word"]:
            issues.append("Backward match")

        gap = match["start_word"] - previous_match["end_word"]

        if gap > 40:
            issues.append("Possible missing passage")

    # First section needs special handling
    if section == "section_000":
        issues.append("Introduction and PDF extraction errors")

    status = "review" if issues else "candidate"

    alignment_plan.append({
        "section": section,
        "audio_path": str(audio_path),
        "book_start_word": match["start_word"],
        "book_end_word": match["end_word"],
        "proposed_text": match["book_text"],
        "transcription": match["transcription"],
        "similarity": score,
        "status": status,
        "issues": issues
    })

    previous_match = match

# Save the plan
plan_path = Path("../../data/test/alignment_plan.json")

plan_path.write_text(
    json.dumps(alignment_plan, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Show summary
for item in alignment_plan:
    print(
        item["section"],
        "|", item["status"],
        "|", item["similarity"], "%"
    )

print("\nSaved:", plan_path)

section_000 | review | 82.35 %
section_001 | candidate | 98.65 %
section_002 | candidate | 95.03 %
section_003 | candidate | 98.26 %
section_004 | candidate | 96.54 %
section_005 | candidate | 98.93 %
section_006 | candidate | 97.39 %
section_007 | candidate | 98.91 %
section_008 | candidate | 93.84 %
section_009 | candidate | 94.55 %

Saved: ../../data/test/alignment_plan.json


In [8]:
first = matches[0]

audio_text = normalize(first["transcription"])
book_text = normalize(first["book_text"])

anchor = "bir podsho o'tgan"

audio_pos = audio_text.find(anchor)
book_pos = book_text.find(anchor)

if audio_pos != -1 and book_pos != -1:

    introduction = audio_text[:audio_pos]

    print("AUDIO BEFORE ANCHOR:")
    print(introduction)

    print("\nBOOK BEFORE ANCHOR:")
    print(book_text[:book_pos])

    print("\nAUDIO AFTER ANCHOR:")
    print(audio_text[audio_pos:audio_pos + 250])
else:
    print("Anchor not found in both texts")

AUDIO BEFORE ANCHOR:
qadrli bolajonlar o'zbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman bolalar adabiyoti jurnali qadim zamonda 

BOOK BEFORE ANCHOR:
ain ochilar y adim zamonda 

AUDIO AFTER ANCHOR:
bir podsho o'tgan ekan u taxtda doim g'amgin o'tirar ekan kunlarning birida o'ng qo'li vaziri undan podshohim nimagadir mudom g'amginsiz boisi nima deb so'rabdi mol u davlatim yetarli ammo o'lgandan keyin bel bog'lab bo'zlaydigan o'g'lim yo'q o'g'il 


In [9]:
# Directory for alignment text drafts
draft_dir = Path("../../data/test/alignment_drafts")
draft_dir.mkdir(parents=True, exist_ok=True)

alignment_candidates = []

for match in matches:
    section = match["section"]
    score = match["similarity"]

    # Choose initial text source
    if score < 85:
        text = match["transcription"]
        source = "transcription"
    else:
        text = match["book_text"]
        source = "book"

    # Save one text file per section
    text_path = draft_dir / f"{section}.txt"

    # Don't overwrite manual corrections
    if not text_path.exists():
        text_path.write_text(
            text.strip(),
            encoding="utf-8"
        )

    alignment_candidates.append({
        "section": section,
        "audio_path": str(sections_dir / f"{section}.wav"),
        "text_path": str(text_path),
        "source": source,
        "similarity": score,
        "verified": False
    })

    print(
        section,
        "| Source:", source,
        "| Similarity:", score
    )

# Save the candidate manifest
manifest_path = Path(
    "../../data/test/alignment_candidates.json"
)

manifest_path.write_text(
    json.dumps(
        alignment_candidates,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nSaved:", manifest_path)

section_000 | Source: transcription | Similarity: 82.35
section_001 | Source: book | Similarity: 98.65
section_002 | Source: book | Similarity: 95.03
section_003 | Source: book | Similarity: 98.26
section_004 | Source: book | Similarity: 96.54
section_005 | Source: book | Similarity: 98.93
section_006 | Source: book | Similarity: 97.39
section_007 | Source: book | Similarity: 98.91
section_008 | Source: book | Similarity: 93.84
section_009 | Source: book | Similarity: 94.55

Saved: ../../data/test/alignment_candidates.json


In [10]:
for item in alignment_candidates:

    text_path = Path(item["text_path"])

    text = text_path.read_text(
        encoding="utf-8"
    )

    print("\n", item["section"])
    print("Words:", len(text.split()))
    print("Text:", text[:120], "...")
    print("Verified:", item["verified"])


 section_000
Words: 63
Text: Qadrli bolajonlar, o'zbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.  ...
Verified: False

 section_001
Words: 100
Text: bor chinmochin mamlakatida bir daraxt bor u daraxti baxt deb ataladi uning bargini ming dardga shifo deydilar shu bargda ...
Verified: False

 section_002
Words: 87
Text: daraxtdan barg keltirib beraman degan niyat bilan huzuringizga keldim debdi podsho talabgor chiqqanidan juda xursand bo' ...
Verified: False

 section_003
Words: 90
Text: yol yursa ham mo'l yuribdi daryolardan changalzorlardan cho'llardan baland tog'lardan o'tibdi nihoyat bog'i zeboga yetib ...
Verified: False

 section_004
Words: 121
Text: xuroson podshosining qizi bo'laman ismim gulnora devlar meni bu yerga o'g'irlab olib kelgan bu yer devlarning makoni hal ...
Verified: False

 section_005
Words: 97
Text: beribdi yigit gulnoradan birga ketishni so'rabdi gulnora devlar izimizdan borib bizni sog' qo'yishmaydi unda pods